In [8]:
import csv

def ler_transacoes(nome_arquivo):
    lista_transacoes = []

    # tentar abrir o arquivo com segurança (try/except)
    try:
        # O 'with open' fecha o arquivo automaticamente no final
        with open(nome_arquivo, mode='r', encoding='utf-8') as arquivo:
          # ponto exigido
            leitor = csv.DictReader(arquivo)
            for linha in leitor:
                lista_transacoes.append(linha)

    except FileNotFoundError:# ponto exigido
        print(f"❌ Erro: O arquivo '{nome_arquivo}' não foi encontrado. Verifique o nome ou faça o upload.")
    return lista_transacoes

# Vamos executar a função para ver se ela acha e lê o seu arquivo!

print("Testando a leitura do arquivo...\n")
transacoes_brutas = ler_transacoes('transacoes.csv')

# Se a lista tiver dados, mostramos quantas linhas foram lidas e a primeira delas
if transacoes_brutas:
    print(f"✅ Sucesso! Foram lidas {len(transacoes_brutas)} linhas (sem contar o cabeçalho).")
    print("\nVeja como a primeira linha ficou guardada na memória:")
    print(transacoes_brutas[0])

Testando a leitura do arquivo...

✅ Sucesso! Foram lidas 75 linhas (sem contar o cabeçalho).

Veja como a primeira linha ficou guardada na memória:
{'id': '1', 'data': '2026-01-05', 'cliente_id': 'CLI001', 'tipo': 'credito', 'valor': '3500.00', 'descricao': 'Salário janeiro', 'categoria': 'salario'}


In [9]:
from datetime import datetime

def validar_transacao(linha):
    # 1. Valida ID: deve existir e conter apenas números
    if not linha['id'] or not linha['id'].isdigit():
        return None

    # 2. Valida Cliente: não pode ser vazio
    if not linha['cliente_id']:
        return None

    # 3. Valida Tipo: deve ser estritamente 'credito' ou 'debito'
    if linha['tipo'] not in ['credito', 'debito']:
        return None

    # 4. Valida e converte Valor usando try/except
    try:
        valor_float = float(linha['valor'])
        if valor_float <= 0:
            return None
        linha['valor'] = valor_float
    except ValueError:
        return None

    # 5. Valida e converte Data usando try/except
    try:
        linha['data'] = datetime.strptime(linha['data'], '%Y-%m-%d')
    except ValueError:
        return None

    return linha

transacoes_validas = []
total_lidas = len(transacoes_brutas)

print("Iniciando limpeza de dados...\n")

for transacao in transacoes_brutas:
    transacao_limpa = validar_transacao(transacao)

    if transacao_limpa:
        transacoes_validas.append(transacao_limpa)

total_validas = len(transacoes_validas)
total_invalidas = total_lidas - total_validas

# Exibição do resumo (ponto exigido)
print(f"Total de linhas lidas: {total_lidas}")
print(f"Linhas válidas: {total_validas}")
print(f"Linhas inválidas: {total_invalidas}")

Iniciando limpeza de dados...

Total de linhas lidas: 75
Linhas válidas: 68
Linhas inválidas: 7


In [10]:
def gerar_relatorio(transacoes):
    resumo_mensal = {}

    # Agrupamento por mês
    for t in transacoes:
        # Extrai apenas o Ano-Mês da data (ex: 2026-01)
        mes = t['data'].strftime('%Y-%m')
        valor = t['valor']

        # Se o mês ainda não existe no nosso resumo, criamos a estrutura inicial dele
        if mes not in resumo_mensal:
            resumo_mensal[mes] = {
                "quantidade": 0,
                "total_credito": 0.0,
                "total_debito": 0.0,
                "maior_valor": valor,
                "menor_valor": valor
            }

        # Adiciona +1 na quantidade de transações daquele mês
        resumo_mensal[mes]["quantidade"] += 1

        # Soma os valores dependendo se é crédito ou débito
        if t['tipo'] == 'credito':
            resumo_mensal[mes]["total_credito"] += valor
        else:
            resumo_mensal[mes]["total_debito"] += valor

        # Atualiza os maiores e menores valores do mês
        if valor > resumo_mensal[mes]["maior_valor"]:
            resumo_mensal[mes]["maior_valor"] = valor
        if valor < resumo_mensal[mes]["menor_valor"]:
            resumo_mensal[mes]["menor_valor"] = valor

    # 2. Cálculos finais (Saldo e Média) para cada mês agrupadod
    for mes, dados in resumo_mensal.items():
        dados["saldo"] = dados["total_credito"] - dados["total_debito"]
        dados["media"] = (dados["total_credito"] + dados["total_debito"]) / dados["quantidade"]

    # 3. Cálculo de dias entre a primeira e a última transação
    # Pega todas as datas da lista, descobre a menor (mais antiga) e a maior (mais recente)
    datas = [t['data'] for t in transacoes]
    dias_analisados = (max(datas) - min(datas)).days

    # A função retorna o resumo de todos os meses e a quantidade de dias
    return resumo_mensal, dias_analisados



resumo, dias = gerar_relatorio(transacoes_validas)

print("Cálculos concluídos com sucesso!\n")
print(f"Período analisado engloba {dias} dias no total.")
print(f"Meses identificados nos dados: {list(resumo.keys())}")

Cálculos concluídos com sucesso!

Período analisado engloba 114 dias no total.
Meses identificados nos dados: ['2026-01', '2026-02', '2026-03', '2026-04']


In [11]:
def formatar_moeda(valor):
    # Dica do professor: formata para padrão brasileiro (ex: 1.500,50)
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

def exibir_relatorio_mensal(resumo_mensal):
    # Passa por cada mês e exibe os dados formatados
    for mes, dados in resumo_mensal.items():
        print("\n===== RELATÓRIO MENSAL =====")
        print(f"Mês: {mes}")
        print(f"  Transações:    {dados['quantidade']}")
        print(f"  Total crédito: {formatar_moeda(dados['total_credito'])}")
        print(f"  Total débito:  {formatar_moeda(dados['total_debito'])}")
        print(f"  Saldo:         {formatar_moeda(dados['saldo'])}")
        print(f"  Média:         {formatar_moeda(dados['media'])}")
        print(f"  Maior valor:   {formatar_moeda(dados['maior_valor'])}")
        print(f"  Menor valor:   {formatar_moeda(dados['menor_valor'])}")

exibir_relatorio_mensal(resumo)


===== RELATÓRIO MENSAL =====
Mês: 2026-01
  Transações:    16
  Total crédito: R$ 18.000,00
  Total débito:  R$ 1.032,50
  Saldo:         R$ 16.967,50
  Média:         R$ 1.189,53
  Maior valor:   R$ 12.000,00
  Menor valor:   R$ 12,00

===== RELATÓRIO MENSAL =====
Mês: 2026-02
  Transações:    18
  Total crédito: R$ 20.830,00
  Total débito:  R$ 2.230,00
  Saldo:         R$ 18.600,00
  Média:         R$ 1.281,11
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 40,00

===== RELATÓRIO MENSAL =====
Mês: 2026-03
  Transações:    18
  Total crédito: R$ 7.500,00
  Total débito:  R$ 3.594,90
  Saldo:         R$ 3.905,10
  Média:         R$ 616,38
  Maior valor:   R$ 3.500,00
  Menor valor:   R$ 35,00

===== RELATÓRIO MENSAL =====
Mês: 2026-04
  Transações:    16
  Total crédito: R$ 16.400,00
  Total débito:  R$ 2.695,00
  Saldo:         R$ 13.705,00
  Média:         R$ 1.193,44
  Maior valor:   R$ 11.000,00
  Menor valor:   R$ 15,00


In [12]:
LIMITE_SUSPEITO = 10000.00

def identificar_suspeitas(transacoes):
    suspeitas = []
    for t in transacoes:
        if t['valor'] > LIMITE_SUSPEITO:
            suspeitas.append({
                'id': t['id'],
                'cliente_id': t['cliente_id'],
                'data': t['data'].strftime('%Y-%m-%d'),
                'valor': t['valor']
            })
    return suspeitas

def exibir_suspeitas(suspeitas):
    print("\n===== TRANSAÇÕES SUSPEITAS =====")

    if not suspeitas:
        print("Nenhuma transação suspeita encontrada.")
    else:
        for s in suspeitas:
            # Reutilizando a formatação de moeda da etapa anterior
            valor_fmt = f"R$ {s['valor']:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
            print(f"ID: {s['id']} | Cliente: {s['cliente_id']} | Data: {s['data']} | Valor: {valor_fmt}")


lista_suspeitas = identificar_suspeitas(transacoes_validas)
exibir_suspeitas(lista_suspeitas)


===== TRANSAÇÕES SUSPEITAS =====
ID: 5 | Cliente: CLI003 | Data: 2026-02-14 | Valor: R$ 15.000,00
ID: 12 | Cliente: CLI006 | Data: 2026-01-17 | Valor: R$ 12.000,00
ID: 23 | Cliente: CLI001 | Data: 2026-04-02 | Valor: R$ 11.000,00


In [13]:
import json
from datetime import datetime

def salvar_json(total_validas, total_invalidas, resumo_mensal, suspeitas):
    # Montamos a estrutura exata exigida pelo desafio
    dados_relatorio = {
        "gerado_em": datetime.now().strftime('%Y-%m-%d'),
        "total_transacoes_validas": total_validas,
        "total_transacoes_invalidas": total_invalidas,
        "resumo_mensal": resumo_mensal,
        "transacoes_suspeitas": suspeitas
    }

    # Criamos e salvamos o arquivo JSON
    with open('relatorio.json', mode='w', encoding='utf-8') as arquivo:
        json.dump(dados_relatorio, arquivo, ensure_ascii=False, indent=2)

    print("\n✅ Arquivo 'relatorio.json' salvo com sucesso!")


salvar_json(total_validas, total_invalidas, resumo, lista_suspeitas)


✅ Arquivo 'relatorio.json' salvo com sucesso!


In [14]:
def exibir_cabecalho(transacoes_validas, total_lidas):
    total_validas = len(transacoes_validas)
    total_invalidas = total_lidas - total_validas

    # Busca a data mais antiga e a mais recente
    datas = [t['data'] for t in transacoes_validas]
    data_antiga = min(datas).strftime('%d/%m/%Y')
    data_recente = max(datas).strftime('%d/%m/%Y')

    print("========================================")
    print("      RELATÓRIO FINANCEIRO CLEARBANK    ")
    print("========================================")
    print(f"Período analisado: {data_antiga} → {data_recente}")
    print(f"Total de linhas lidas: {total_lidas}")
    print(f"Transações válidas: {total_validas}")
    print(f"Transações inválidas: {total_invalidas}")
    print("========================================")

# ==========================================
# 🚀 CÉLULA DE EXECUÇÃO PRINCIPAL
# Aqui juntamos todo o projeto do início ao fim!
# ==========================================

print("Iniciando processamento de dados...\n")

# 1. Leitura
transacoes_brutas = ler_transacoes('transacoes.csv')
total_lidas = len(transacoes_brutas)

# 2. Limpeza e Validação
transacoes_validas = []
for t in transacoes_brutas:
    t_limpa = validar_transacao(t)
    if t_limpa:
        transacoes_validas.append(t_limpa)

# 3. Cálculos e Suspeitas
resumo_mensal, _ = gerar_relatorio(transacoes_validas)
suspeitas = identificar_suspeitas(transacoes_validas)

# 4. Salvar JSON
salvar_json(len(transacoes_validas), total_lidas - len(transacoes_validas), resumo_mensal, suspeitas)

# 5. Exibição Formatada no Terminal (Atendendo aos requisitos da imagem)
print("\n")
exibir_cabecalho(transacoes_validas, total_lidas)
exibir_relatorio_mensal(resumo_mensal)
exibir_suspeitas(suspeitas)

Iniciando processamento de dados...


✅ Arquivo 'relatorio.json' salvo com sucesso!


      RELATÓRIO FINANCEIRO CLEARBANK    
Período analisado: 02/01/2026 → 26/04/2026
Total de linhas lidas: 75
Transações válidas: 68
Transações inválidas: 7

===== RELATÓRIO MENSAL =====
Mês: 2026-01
  Transações:    16
  Total crédito: R$ 18.000,00
  Total débito:  R$ 1.032,50
  Saldo:         R$ 16.967,50
  Média:         R$ 1.189,53
  Maior valor:   R$ 12.000,00
  Menor valor:   R$ 12,00

===== RELATÓRIO MENSAL =====
Mês: 2026-02
  Transações:    18
  Total crédito: R$ 20.830,00
  Total débito:  R$ 2.230,00
  Saldo:         R$ 18.600,00
  Média:         R$ 1.281,11
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 40,00

===== RELATÓRIO MENSAL =====
Mês: 2026-03
  Transações:    18
  Total crédito: R$ 7.500,00
  Total débito:  R$ 3.594,90
  Saldo:         R$ 3.905,10
  Média:         R$ 616,38
  Maior valor:   R$ 3.500,00
  Menor valor:   R$ 35,00

===== RELATÓRIO MENSAL =====
Mês: 2026-04
  Transa